In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [2]:
df_features = pd.read_csv("../data/processed/cleaned_waitlist.csv")
df_features.head() 

,ON_DIALYSIS,A2A2B_ELIGIBILITY,GENDER,ABO,BMI_TCR,FUNC_STAT_TCR,INIT_STAT,INIT_CPRA,INIT_AGE,INIT_DATE,...,DIAG_KI,MULTIORG,LISTING_CTR_CODE,outcome,event_adverse,event_transplant,censored,days_to_event,DIALYSIS_DURATION,DIALYSIS_AFTER_LISTING
0,Y,-1.0,F,B,31.63,2080.0,4099,0.0,53,2020-03-25,...,-1.0,N,13609,died,1,0,0,287.0,726.0,0
1,Y,-1.0,M,A,30.04,2070.0,4099,0.0,56,2020-02-14,...,-1.0,N,6975,removed_administrative,0,0,0,2322.0,912.0,0
2,N,-1.0,F,O,32.85,2070.0,4099,0.0,47,2020-05-27,...,-1.0,N,19716,died,1,0,0,604.0,0.0,0
3,Y,-1.0,M,A,20.00,2090.0,4099,0.0,61,2020-04-02,...,-1.0,N,8587,removed_too_sick,1,0,0,909.0,453.0,0
4,Y,-1.0,M,AB,23.30,2070.0,4010,0.0,61,2020-02-05,...,-1.0,N,18352,removed_too_sick,1,0,0,231.0,391.0,0


In [3]:
# Select only the recommended features per data_dictionary.md
feature_cols = [
    'ON_DIALYSIS', 'A2A2B_ELIGIBILITY', 'GENDER', 'ABO', 'BMI_TCR',
    'FUNC_STAT_TCR', 'INIT_STAT', 'INIT_CPRA', 'INIT_AGE',
    'DIALYSIS_DURATION', 'DIALYSIS_AFTER_LISTING', 'ETHCAT', 'REGION'
]

df_model = df_features[feature_cols].copy()
df_model.head()

,ON_DIALYSIS,A2A2B_ELIGIBILITY,GENDER,ABO,BMI_TCR,FUNC_STAT_TCR,INIT_STAT,INIT_CPRA,INIT_AGE,DIALYSIS_DURATION,DIALYSIS_AFTER_LISTING,ETHCAT,REGION
0,Y,-1.0,F,B,31.63,2080.0,4099,0.0,53,726.0,0,2,8
1,Y,-1.0,M,A,30.04,2070.0,4099,0.0,56,912.0,0,1,5
2,N,-1.0,F,O,32.85,2070.0,4099,0.0,47,0.0,0,1,11
3,Y,-1.0,M,A,20.00,2090.0,4099,0.0,61,453.0,0,5,7
4,Y,-1.0,M,AB,23.30,2070.0,4010,0.0,61,391.0,0,2,7


In [4]:
df_model['DIALYSIS_DURATION'].describe()

count    494803.000000
mean        582.210546
std         918.094631
min           0.000000
25%           0.000000
50%         247.000000
75%         762.000000
max       15356.000000
Name: DIALYSIS_DURATION, dtype: float64

In [5]:
# log-transform to compress skew in DIALYSIS_DURATION
# (unsigned, since DIALYSIS_DURATION is already clipped to 0 in outliers, so no neg left)
df_model['DIALYSIS_DURATION_LOG'] = np.log1p(df_model['DIALYSIS_DURATION'])
df_model['DIALYSIS_DURATION_LOG'].describe()

count    494803.000000
mean          4.032315
std           3.141309
min           0.000000
25%           0.000000
50%           5.513429
75%           6.637258
max           9.639327
Name: DIALYSIS_DURATION_LOG, dtype: float64

In [6]:
categorical_cols = [
    'ON_DIALYSIS', 'GENDER', 'ABO', 'A2A2B_ELIGIBILITY',
    'INIT_STAT', 'ETHCAT', 'REGION'
]

numerical_cols = ['BMI_TCR', 'INIT_CPRA', 'INIT_AGE', 'DIALYSIS_DURATION', 'DIALYSIS_DURATION_LOG']

# DIALYSIS_AFTER_LISTING is a 0/1 flag from outliers.ipynb
# FUNC_STAT_TCR needs special decoding (per data_dictionary.md) not simple encoding or scaling

In [7]:
#Caps BMI at a maximum of 80, there was a data entry error
df_model.loc[df_model['BMI_TCR'] > 80, 'BMI_TCR'] = df_model['BMI_TCR'].median()
df_model['BMI_TCR'].describe()

count    494803.000000
mean         28.817576
std           5.733061
min          12.405000
25%          24.660000
50%          28.530000
75%          32.830000
max          45.085000
Name: BMI_TCR, dtype: float64

In [8]:
#encode categorical data
new_df = pd.get_dummies(df_model, columns=categorical_cols, drop_first=True)
new_df.head()

,BMI_TCR,FUNC_STAT_TCR,INIT_CPRA,INIT_AGE,DIALYSIS_DURATION,DIALYSIS_AFTER_LISTING,DIALYSIS_DURATION_LOG,ON_DIALYSIS_Y,GENDER_M,ABO_A1,...,REGION_2,REGION_3,REGION_4,REGION_5,REGION_6,REGION_7,REGION_8,REGION_9,REGION_10,REGION_11
0,31.63,2080.0,0.0,53,726.0,0,6.588926,True,False,False,...,False,False,False,False,False,False,True,False,False,False
1,30.04,2070.0,0.0,56,912.0,0,6.816736,True,True,False,...,False,False,False,True,False,False,False,False,False,False
2,32.85,2070.0,0.0,47,0.0,0,0.000000,False,False,False,...,False,False,False,False,False,False,False,False,False,True
3,20.00,2090.0,0.0,61,453.0,0,6.118097,True,True,False,...,False,False,False,False,False,True,False,False,False,False
4,23.30,2070.0,0.0,61,391.0,0,5.971262,True,True,False,...,False,False,False,False,False,True,False,False,False,False


In [9]:
#Check INIT_CPRA range/outliers and most frequent values (which is 0.00)
print(df_model['INIT_CPRA'].describe())
df_model['INIT_CPRA'].value_counts().head(10)

count    494803.000000
mean          6.962676
std          21.382831
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max         100.000000
Name: INIT_CPRA, dtype: float64


INIT_CPRA
0.00      420160
0.03         904
16.56        586
0.26         583
100.00       568
99.99        542
56.09        500
2.32         499
50.01        496
0.02         474
Name: count, dtype: int64

In [10]:
#normalize numerical data
scaler = StandardScaler()
new_df[numerical_cols] = scaler.fit_transform(new_df[numerical_cols])
new_df[numerical_cols].head()

,BMI_TCR,INIT_CPRA,INIT_AGE,DIALYSIS_DURATION,DIALYSIS_DURATION_LOG
0,0.490563,-0.32562,0.077090,0.156617,0.813869
1,0.213224,-0.32562,0.282551,0.359211,0.886390
2,0.703364,-0.32562,-0.333834,-0.634152,-1.283643
3,-1.538024,-0.32562,0.624987,-0.140738,0.663986
4,-0.962415,-0.32562,0.624987,-0.208269,0.617242


In [11]:
new_df['DIALYSIS_DURATION'].describe()

count    4.948030e+05
mean     3.676189e-18
std      1.000001e+00
min     -6.341516e-01
25%     -6.341516e-01
50%     -3.651158e-01
75%      1.958291e-01
max      1.609181e+01
Name: DIALYSIS_DURATION, dtype: float64

In [12]:
new_df['DIALYSIS_DURATION_LOG'].describe()

count    4.948030e+05
mean    -3.676189e-18
std      1.000001e+00
min     -1.283643e+00
25%     -1.283643e+00
50%      4.714961e-01
75%      8.292547e-01
max      1.784930e+00
Name: DIALYSIS_DURATION_LOG, dtype: float64

In [13]:
new_df.to_csv("../data/processed/task3_encoded_scaled.csv", index=False)

Done: encoding + normalizing for ON_DIALYSIS, GENDER, ABO, A2A2B_ELIGIBILITY, INIT_STAT, ETHCAT, REGION, BMI_TCR, INIT_CPRA, INIT_AGE, DIALYSIS_DURATION

DIALYSIS_DATE / INIT_DATE: converted to DIALYSIS_DURATION + DIALYSIS_AFTER_LISTING in outliers.ipynb, encoded/scaled here. 

DIALYSIS_DURATION_LOG is the log-transformed version. This version would help make extreme values scale proportionally and not swing predictions in logistic regression, KNN, or neural nets.

Errors: BMI_TCR had one outlier value of 430,226 (data entry error), but now capped at 80.

Still need:
- FUNC_STAT_TCR: needs decoding (mixed Karnofsky/Lansky scales, see data_dictionary.md)